# Build a `MarketContext` from raw quotes — canonical path tour

This notebook walks through the analyst-facing flow for building a
`MarketContext` from a JSON `CalibrationEnvelope`. It is the canonical
entry point — `calibrate(envelope_json).market` — and the same surface is
exposed in Rust (`finstack_valuations::calibration::api::engine::execute`)
and JavaScript (`valuations.calibrate(envelopeJson)`).

We load one of the reference envelopes shipped under
`finstack/valuations/examples/market_bootstrap/`, run it through the
calibration engine, verify residuals, and access the resulting context.

In [ ]:
import json
from pathlib import Path

from finstack.valuations import calibrate

# Path to the reference envelope. Adjust if running outside the repo root.
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "finstack" / "valuations" / "examples" / "market_bootstrap").exists():
    if REPO_ROOT == REPO_ROOT.parent:
        raise RuntimeError("could not locate the repo root from cwd")
    REPO_ROOT = REPO_ROOT.parent

envelope_path = (
    REPO_ROOT
    / "finstack"
    / "valuations"
    / "examples"
    / "market_bootstrap"
    / "01_usd_discount.json"
)
envelope_json = envelope_path.read_text()
envelope = json.loads(envelope_json)
print(f"schema: {envelope['schema']}")
print(f"plan id: {envelope['plan']['id']}")
print(f"steps: {[step['id'] for step in envelope['plan']['steps']]}")

In [ ]:
result = calibrate(envelope_json)
print(f"success: {result.success}")
print(f"rmse: {result.rmse:.3e}")
print(f"max residual: {result.max_residual:.3e}")
print(f"iterations: {result.iterations}")
result.report_to_dataframe()

In [ ]:
# `result.market` is the calibrated MarketContext, ready for downstream use.
market = result.market

# Look up the discount curve we just built.
curve = market.get_discount("USD-OIS")
print(f"discount factor at t=0:  {curve.df(0.0):.6f}")
print(f"discount factor at t=1y: {curve.df(1.0):.6f}")
print(f"discount factor at t=5y: {curve.df(5.0):.6f}")

## Persisting the materialized market

`result.market_json` returns the same context as a JSON snapshot. This is
useful for caching a calibrated market between processes or for diff'ing
two calibrated states.

`MarketContext` round-trips through this snapshot losslessly:
deserialize via `MarketContext.from_json(...)` (Python) or
`MarketContext::try_from(state)` (Rust).

In [ ]:
# Round-trip via the materialized JSON snapshot.
snapshot_json = result.market_json
print(f"snapshot length: {len(snapshot_json)} bytes")
# Phase 2 of the notebook tour will demonstrate composing markets and
# pulling FX cross rates / bond prices / equity spots from initial_market.